# V9 - Fast LoRA compact multi-equation reasoning

Single-stage Fast LoRA. Targets contain all compact equation spans extracted from `response_vi`, then a final numeric answer line. Checkpoint selection uses only a source-disjoint heldout split from train.

In [ ]:
# ============================================================
# 0. Install/import dependencies
# ============================================================
import os, sys, json, math, time, re, random, hashlib, inspect, shutil, unicodedata
from pathlib import Path
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

PIP_INSTALL_DEPS = False  # Kaggle official run: Internet OFF; avoid pip overhead
PIP_PACKAGES = ["peft", "accelerate", "datasets", "evaluate"]

if PIP_INSTALL_DEPS:
    import subprocess
    print("[pip] Installing:", " ".join(PIP_PACKAGES))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES])

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
import peft

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
print("PEFT :", getattr(peft, "__version__", "unknown"))


In [ ]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Cannot find any path: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/dataset-math",
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "dataset",
)

MODEL_NAME = str(first_existing(
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "GPT2_vietnamese",
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"

# v9_compact_equation: compact reasoning SFT. valid.json is report-only; checkpoint
# selection uses a source-disjoint heldout split from train.json.
USE_KD = False
REQUIRE_KD_FILE = False

# Run mode: "phase1" reports source-heldout + valid.json; "phase2" writes test_predictions.json.
RUN_MODE = "phase1"

# Prompt & special tokens
PROMPT_TEMPLATE = "Bài toán: {q}\nLời giải: "
SAFE_EOS_ID = 50256
N_POSITIONS = 1024

# Working dirs / outputs
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
STAGE_A_OUTPUT_DIR = WORKING_DIR / "gpt2_math_lora_v9_stage_a"
SFT_OUTPUT_DIR     = WORKING_DIR / "gpt2_math_lora_v9_stage_b_sft"
STAGE_C_OUTPUT_DIR = WORKING_DIR / "gpt2_math_lora_v9_stage_c_mixed"
FINAL_OUTPUT_DIR   = WORKING_DIR / "gpt2_math_lora_v9_final"

SOURCE_VALID_OUTPUT_PATH        = WORKING_DIR / "source_valid_output.json"
SOURCE_VALID_REPORT_PATH        = WORKING_DIR / "source_valid_report.json"
STAGE_A_VALID_OUTPUT_PATH       = WORKING_DIR / "valid_output_stage_a.json"
STAGE_A_VALID_REPORT_PATH       = WORKING_DIR / "valid_report_stage_a.json"
SFT_VALID_OUTPUT_PATH           = WORKING_DIR / "valid_output_sft.json"
SFT_VALID_REPORT_PATH           = WORKING_DIR / "valid_report_sft.json"
VALID_OUTPUT_PATH               = WORKING_DIR / "valid_output.json"
VALID_REPORT_PATH               = WORKING_DIR / "valid_report.json"
VALID_OVERLAP_AUDIT_PATH        = WORKING_DIR / "valid_overlap_audit.json"
COMPACT_EQUATION_COVERAGE_PATH  = WORKING_DIR / "compact_equation_coverage.json"
RL_LOG_PATH                     = WORKING_DIR / "rl_training_log.jsonl"
RL_SUMMARY_PATH                 = WORKING_DIR / "rl_reward_summary.json"
TEST_OUTPUT_PATH                = WORKING_DIR / "test_predictions.json"

# Checkpoint selection. The key change versus V8: do not select on valid.json.
CHECKPOINT_ROOT_DIR              = WORKING_DIR / "gpt2_math_lora_v9_checkpoints"
CHECKPOINT_EVAL_DIR              = WORKING_DIR / "v9_checkpoint_eval_source_valid"
CHECKPOINT_SELECTION_REPORT_PATH = WORKING_DIR / "checkpoint_selection_report.json"
SELECTED_CHECKPOINT_INFO_PATH    = WORKING_DIR / "selected_checkpoint_info.json"
SELECT_CHECKPOINTS_ON_SOURCE_VALID = True
SOURCE_VALID_GROUP_FRACTION      = 0.10
SOURCE_VALID_MAX_EVAL_RECORDS    = 1000
SOURCE_GROUP_KEY_FIELDS          = ["original_question_en", "original_question_vi", "query_vi"]
CHECKPOINT_EVAL_NUM_BEAMS        = 2
CHECKPOINT_EVAL_MAX_NEW_TOKENS   = 96
KEEP_CHECKPOINT_EVAL_OUTPUTS     = True
CHECKPOINT_TIE_BREAK             = "earlier_epoch"  # earlier_epoch or later_epoch

# Smoke/debug knobs. For a fast local smoke run, set these small.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
DROP_EXACT_DUPLICATES = True
DROP_NON_EXTRACTABLE = True

# Stage A
STAGE_A_NAME = "stage_a_compact_equation_lora"
STAGE_A_TARGET_MODE = "compact_equation"  # answer_only or compact_equation
STAGE_A_EPOCHS = 8.0
STAGE_A_LR = 1e-3
MAX_LENGTH_STAGE_A = 256

# Stage B
RUN_STAGE_B = False
STAGE_B_NAME = "stage_b_compact_equation_lora"
STAGE_B_TARGET_MODE = "compact_equation"
STAGE_B_MAX_RECORDS = None
STAGE_B_EPOCHS = 0.0
STAGE_B_LR = 0.0
MAX_LENGTH_STAGE_B = 256

# Stage C
RUN_STAGE_C = False
STAGE_C_NAME = "stage_c_mixed_replay_lora"
STAGE_C_TARGET_MODE = "mixed_replay"
STAGE_C_MAX_RECORDS = None
STAGE_C_EPOCHS = 0.0
STAGE_C_LR = 0.0
MAX_LENGTH_STAGE_C = 256
MIXED_COMPACT_RATIO = 0.70

# Compact-equation target
LOCAL_REASON_MAX_TOKENS = 160
COMPACT_EQUATION_JOINER = "; "
COMPACT_EQUATION_KEEP_ALL = True

# Weighted loss. Equation text is normal loss; final anchor and final numeric
# answer are emphasized so the model still optimizes the scoring-critical tail.
EQUATION_TOKEN_WEIGHT = 1.0
FINAL_ANCHOR_TOKEN_WEIGHT = 1.5
FINAL_ANSWER_TOKEN_WEIGHT = 3.0
EOS_TOKEN_WEIGHT = 1.0

# Trainer
PER_DEVICE_BATCH_SIZE = 16  # fallback: 8 if OOM
GRAD_ACCUM = 2              # fallback: 4 if PER_DEVICE_BATCH_SIZE=8
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 42

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["c_attn", "c_proj", "c_fc"]

# Stage C RL remains disabled; V10 curriculum is SFT-only.
RL_ENABLED = False
RL_MAX_PROMPTS = 6000
RL_GROUP_SIZE = 2
RL_PROMPT_BATCH_SIZE = 4
RL_MAX_NEW_TOKENS = 64
RL_LR = 5e-6
RL_MAX_STEPS = 0
RL_TEMPERATURE = 0.8
RL_TOP_P = 0.9
RL_SFT_REPLAY_COEF = 0.20
RL_GRAD_CLIP = 1.0

# Generation defaults for evaluation/submission
MAX_NEW_TOKENS = 96
NUM_BEAMS = 2
DECODE_BATCH_SIZE = 8
NO_REPEAT_NGRAM = 4
REPETITION_PENALTY = 1.15
LENGTH_PENALTY = 0.9
USE_TYPE_AWARE_FEWSHOT = False
USE_SELF_CONSISTENCY = False
SANITIZE_TO_ANSWER_ONLY = False
RUN_STAGE_A_GENERATION_EVAL = False
RUN_SFT_GENERATION_EVAL = False
INFER_FP16 = True

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# Speed knobs. Harmless on GPUs that do not support TF32.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("TRAIN_FILE       :", TRAIN_FILE)
print("VALID_FILE       :", VALID_FILE)
print("TEST_FILE        :", TEST_FILE, "| exists:", TEST_FILE.exists())
print("MODEL_NAME       :", MODEL_NAME)
print("RUN_MODE         :", RUN_MODE)
print("FINAL_OUTPUT_DIR :", FINAL_OUTPUT_DIR)
print("CHECKPOINT_ROOT  :", CHECKPOINT_ROOT_DIR)
print("SOURCE_VALID_MAX :", SOURCE_VALID_MAX_EVAL_RECORDS)


In [ ]:
# ============================================================
# 2. Data loading + robust numeric evaluator
# ============================================================
def load_records(path: str | Path) -> list[dict]:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

ANSWER_ANCHORS = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}
EOS_ARTIFACT_RE = re.compile(r"(?:\s*(?:hue|<\|endoftext\|>|</s>|<pad>))+\s*$", re.IGNORECASE)

def clean_decoded_artifacts(text: str | None) -> str:
    """Remove decoded pseudo-EOS artifacts before answer parsing/saving.

    The task asks us to use SAFE_EOS_ID=50256 because the GPT-2 Vietnamese model
    embedding matrix is sized for ids 0..50256. In this tokenizer, however,
    id 50256 decodes to the ordinary string "hue", not to a special token.
    If generation stops on this id, Hugging Face includes it in decoded text.
    Official scoring needs a clean numeric answer, so strip only trailing
    terminator artifacts.
    """
    text = str(text or "").strip()
    for _ in range(4):
        new_text = EOS_ARTIFACT_RE.sub("", text).strip()
        if new_text == text:
            break
        text = new_text
    return text

def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # V3-stable behavior: id 50256 is required as model EOS/PAD but decodes to
    # the ordinary token "hue" in this tokenizer, so strip it by id before
    # decoding rather than relying on skip_special_tokens.
    return clean_decoded_artifacts(tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True))

def _clean_tail(text: str) -> str:
    text = clean_decoded_artifacts(text).split("\n", 1)[0].strip()
    text = re.sub(r"[.,;:。、“”\"')\]]+$", "", text).strip()
    text = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)\b.*$", "", text, flags=re.IGNORECASE)
    return text.strip()

def extract_answer(text: str | None) -> str | None:
    if not text:
        return None
    best_end = -1
    best_tail = None
    for pat in ANSWER_ANCHORS:
        for m in pat.finditer(text):
            if m.end() > best_end:
                best_end = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    boxes = BOXED_RE.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(text: str | None) -> float | None:
    if text is None:
        return None
    value = str(text).strip()
    if not value:
        return None
    if re.fullmatch(r"-?\d+,\d+", value):
        try:
            return float(value.replace(",", "."))
        except ValueError:
            return None
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", value):
        try:
            parsed = float(value)
            return parsed if math.isfinite(parsed) else None
        except ValueError:
            return None
    assignment = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", value)
    if assignment:
        value = assignment.group(1).strip()
    if value.startswith("(") and value.endswith(")") and re.search(r"\d\s*,\s*\d", value):
        return None
    if value.startswith("[") and value.endswith("]"):
        return None
    for _ in range(3):
        new_value = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", value)
        if new_value == value:
            break
        value = new_value
    value = re.sub(r"\\text\{[^}]*\}", "", value)
    value = re.sub(r"\\mathrm\{[^}]*\}", "", value)
    value = value.replace("$", "")
    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        value = value.replace(token, "")
    for token in ("\\cdot", "\\times"):
        value = value.replace(token, "*")
    value = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", value)
    value = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", value)
    value = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", value)
    value = value.replace("\\pi", "pi")
    value = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", value)
    value = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", value)
    value = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", value)
    has_period = "." in value
    comma_count = value.count(",")
    if comma_count == 1 and not has_period and re.search(r"\d,\d", value):
        value = re.sub(r"(?<=\d),(?=\d)", ".", value)
    elif comma_count >= 1:
        value = re.sub(r"(?<=\d),(?=\d{3}\b)", "", value)
    value = re.sub(r"\s+", "", value)
    if not value or "," in value:
        return None
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", value)
    if leftover:
        return None
    try:
        parsed = eval(value.replace("^", "**"), {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None
    if isinstance(parsed, bool):
        return None
    if isinstance(parsed, (int, float)):
        parsed = float(parsed)
        return parsed if math.isfinite(parsed) else None
    return None

def extract_gold(record: dict) -> tuple[str | None, float | None]:
    answer = extract_answer(record.get("response_vi"))
    return answer, parse_number(answer)

def extract_pred(record: dict) -> tuple[str | None, float | None]:
    answer = extract_answer(record.get("model_output"))
    return answer, parse_number(answer)

def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(error_value: float | None, extractable: bool) -> int:
    if not extractable or error_value is None:
        return 0
    if error_value <= 0.01:
        return 10
    if error_value <= 0.10:
        return 5
    if error_value <= 0.50:
        return 1
    return 0

def evaluate_predictions(pred_items: list[dict], gold_items: list[dict]) -> dict:
    if len(pred_items) != len(gold_items):
        raise ValueError(f"Prediction count {len(pred_items)} != gold count {len(gold_items)}")
    rows, total, extractable, numeric_pairs, rel_errors = [], 0, 0, 0, []
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    by_type = defaultdict(lambda: {"n": 0, "raw_score": 0, "extractable": 0, "bucket_10": 0, "bucket_5": 0, "bucket_1": 0, "bucket_0": 0})
    for pred, gold in zip(pred_items, gold_items):
        gold_answer, gold_num = extract_gold(gold)
        pred_answer, pred_num = extract_pred(pred)
        is_extractable = pred_answer is not None
        error_value = rel_error(pred_num, gold_num)
        score = score_one(error_value, is_extractable)
        t = gold.get("type") or pred.get("type") or "unknown"
        total += score
        extractable += int(is_extractable)
        buckets[score] = buckets.get(score, 0) + 1
        if gold_num is not None and pred_num is not None and error_value is not None:
            numeric_pairs += 1
            rel_errors.append(error_value)
        by_type[t]["n"] += 1
        by_type[t]["raw_score"] += score
        by_type[t]["extractable"] += int(is_extractable)
        by_type[t][f"bucket_{score}"] += 1
        rows.append({
            "id": gold.get("id", pred.get("id")),
            "type": t,
            "gold_answer": gold_answer,
            "gold_num": gold_num,
            "pred_answer": pred_answer,
            "pred_num": pred_num,
            "rel_error": error_value,
            "extractable": is_extractable,
            "score": score,
        })
    n = len(rows)
    by_type_final = {}
    for t, d in sorted(by_type.items()):
        d = dict(d)
        d["score_10"] = d["raw_score"] / d["n"] if d["n"] else 0.0
        by_type_final[t] = d
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": 10 * n,
            "score_10": total / n if n else 0.0,
            "score_pct": total / (10 * n) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "by_type": by_type_final,
        "rows": rows,
    }

def save_eval_report(pred_path: Path, gold_records: list[dict], report_path: Path) -> dict:
    pred_items = json.loads(Path(pred_path).read_text(encoding="utf-8"))
    report = evaluate_predictions(pred_items, gold_records)
    Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

def canonicalize_answer(gold_num: float | None) -> str | None:
    if gold_num is None:
        return None
    if abs(gold_num - round(gold_num)) < 1e-9:
        return str(int(round(gold_num)))
    return f"{gold_num:g}"

def sanitize_model_output(text: str | None) -> str:
    """Save a clean answer line if the model produced a parseable answer.

    This is prediction post-processing only: it uses the model's own decoded
    answer, never the gold answer. It prevents harmless decoded terminators or
    extra continuation text from making an otherwise numeric prediction
    unparseable by the official-style scorer.
    """
    cleaned = clean_decoded_artifacts(text)
    pred_answer = extract_answer(cleaned)
    pred_num = parse_number(pred_answer)
    canonical = canonicalize_answer(pred_num)
    if canonical is not None:
        return f"Đáp án là: {canonical}"
    return cleaned

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
test_records_for_info = load_records(TEST_FILE) if TEST_FILE.exists() else []

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records), "| test:", len(test_records_for_info))
print("train type distribution:", dict(Counter(r.get("type") for r in train_records).most_common()))


In [ ]:
# ============================================================
# 3. Source-disjoint split + compact multi-equation targets
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

def stable_fraction(value) -> float:
    h = hashlib.sha256(str(value).encode("utf-8")).hexdigest()
    return int(h[:8], 16) / 0x100000000

def normalize_source_text(text: str | None) -> str:
    text = unicodedata.normalize("NFKC", str(text or "")).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def source_group_key(rec: dict) -> str:
    for field in SOURCE_GROUP_KEY_FIELDS:
        value = normalize_source_text(rec.get(field))
        if value:
            return f"{field}:{value}"
    return f"source_id:{rec.get('_source_id', rec.get('id', 'unknown'))}"

def build_answer_only_target(canonical_answer: str) -> str:
    return f"Đáp án là: {canonical_answer}"

def save_valid_overlap_audit(train_recs: list[dict], valid_recs: list[dict], path: Path):
    train_q = Counter((r.get("query_vi") or "").strip() for r in train_recs)
    train_source = Counter(source_group_key(r) for r in train_recs)
    seen_query = []
    seen_source = []
    by_type = defaultdict(lambda: {"n": 0, "seen_query": 0, "seen_source": 0})
    for i, rec in enumerate(valid_recs):
        q = (rec.get("query_vi") or "").strip()
        sk = source_group_key(rec)
        t = rec.get("type") or "unknown"
        by_type[t]["n"] += 1
        if q in train_q:
            seen_query.append(i)
            by_type[t]["seen_query"] += 1
        if sk in train_source:
            seen_source.append(i)
            by_type[t]["seen_source"] += 1
    report = {
        "train_n": len(train_recs),
        "valid_n": len(valid_recs),
        "train_unique_q": len(train_q),
        "train_unique_source_group": len(train_source),
        "valid_seen_query": len(seen_query),
        "valid_seen_query_pct": len(seen_query) / len(valid_recs) if valid_recs else 0.0,
        "valid_seen_source_group": len(seen_source),
        "valid_seen_source_group_pct": len(seen_source) / len(valid_recs) if valid_recs else 0.0,
        "valid_seen_query_ids_first20": seen_query[:20],
        "valid_seen_source_group_ids_first20": seen_source[:20],
        "valid_by_type": dict(sorted((k, dict(v)) for k, v in by_type.items())),
    }
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[overlap]", json.dumps({k: report[k] for k in ["train_n", "valid_n", "train_unique_source_group", "valid_seen_query", "valid_seen_source_group"]}, ensure_ascii=False))
    return report

def clean_records(records: list[dict], split: str) -> list[dict]:
    seen_exact = set()
    out = []
    dropped_dup = dropped_empty = dropped_non_numeric = 0
    for source_id, rec in enumerate(records):
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_empty += 1
            continue
        if split == "train" and DROP_EXACT_DUPLICATES:
            key = (q, r)
            if key in seen_exact:
                dropped_dup += 1
                continue
            seen_exact.add(key)
        gold_str, gold_num = extract_gold(rec)
        canonical = canonicalize_answer(gold_num)
        if DROP_NON_EXTRACTABLE and canonical is None:
            dropped_non_numeric += 1
            continue
        out.append({
            **rec,
            "query_vi": q,
            "response_vi": r,
            "_source_id": source_id,
            "_gold_str": gold_str,
            "_gold_num": gold_num,
            "_canonical_answer": canonical,
        })
    print(f"[clean:{split}] kept={len(out)} dropped_dup={dropped_dup} dropped_empty={dropped_empty} dropped_non_numeric={dropped_non_numeric}")
    return out

ANSWER_TAIL_RE = re.compile(
    r"(đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?|đ[áa]p\s*[áa]n\s*[:：]|c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?|the\s*answer\s*is\s*[:：]?|####)",
    re.IGNORECASE,
)
FINAL_ANSWER_RE = re.compile(r"(đáp\s*án\s*là\s*[:：]?)(\s*)(.+)$", re.IGNORECASE | re.DOTALL)
LATEX_BOX_RE = re.compile(r"\\boxed\{([^{}]+)\}")
EQUATION_TOKEN = r"(?:\\[A-Za-z]+|[A-Za-z_]+|\d+(?:[.,]\d+)?|[()+\-*/×÷=^:%])"
EQUATION_RUN_RE = re.compile(rf"{EQUATION_TOKEN}(?:\s*{EQUATION_TOKEN}){{2,}}")

def strip_existing_answer_tail(text: str) -> str:
    matches = list(ANSWER_TAIL_RE.finditer(text or ""))
    if matches:
        text = text[:matches[-1].start()]
    text = LATEX_BOX_RE.sub(r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_equation_surface(text: str) -> str:
    text = LATEX_BOX_RE.sub(r"\1", text or "")
    for _ in range(3):
        new_text = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"(\1)/(\2)", text)
        if new_text == text:
            break
        text = new_text
    text = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", text)
    replacements = {
        "\\times": "×",
        "\\cdot": "*",
        "\\div": "÷",
        "\\pi": "pi",
        "$": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_reason_chunks(text: str) -> list[str]:
    # Split on sentence punctuation only when followed by whitespace, so decimal
    # values such as 4.5 or 0,75 are not fragmented.
    raw = re.split(r"(?<=[.!?。])\s+|[\n;]+", text)
    return [s.strip(" -•\t") for s in raw if s.strip(" -•\t")]

def clean_equation_candidate(text: str) -> str:
    text = normalize_equation_surface(text)
    text = re.sub(r"\s*([+\-*/×÷=^:%(),])\s*", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip(" =:+-*/×÷^,.;")
    parts = text.split()
    # Regex runs may catch the ASCII prefix of a Vietnamese word after a
    # number, e.g. "49 qu" from "49 quả". Keep real symbolic tokens such as
    # x, PQ, QR, and pi, but drop trailing prose fragments.
    ops = {"+", "-", "*", "/", "×", "÷", "=", "^", ":", "("}
    while parts and re.fullmatch(r"[A-Za-z_]+", parts[0]) and parts[0].lower() not in {"pi"}:
        nxt = parts[1] if len(parts) >= 2 else ""
        if nxt in ops:
            break
        parts.pop(0)
    while parts and re.fullmatch(r"[A-Za-z_]{2,}", parts[-1]) and parts[-1].lower() not in {"pi"}:
        prev = parts[-2] if len(parts) >= 2 else ""
        if prev in ops:
            break
        parts.pop()
    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_compact_equations(response: str) -> list[str]:
    body = normalize_equation_surface(strip_existing_answer_tail(response))
    equations = []
    seen = set()
    for chunk in split_reason_chunks(body):
        if not (re.search(r"\d", chunk) and re.search(r"[+\-*/×÷=^:]", chunk)):
            continue
        for match in EQUATION_RUN_RE.finditer(chunk):
            cand = clean_equation_candidate(match.group(0))
            if not (re.search(r"\d", cand) and re.search(r"[+\-*/×÷=^:]", cand)):
                continue
            if len(cand) < 5:
                continue
            key = cand.lower()
            if key in seen:
                continue
            equations.append(cand)
            seen.add(key)
    return equations

def cap_text_tokens(text: str, max_tokens: int) -> str:
    if not max_tokens:
        return text.strip()
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text.strip()
    return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True).strip()

def build_compact_equation_target(response: str, canonical_answer: str) -> tuple[str, dict]:
    equations = extract_compact_equations(response)
    if equations:
        eq_text = COMPACT_EQUATION_JOINER.join(equations)
        eq_text = cap_text_tokens(eq_text, LOCAL_REASON_MAX_TOKENS)
        target = f"Phép tính: {eq_text}\nĐáp án là: {canonical_answer}"
    else:
        target = build_answer_only_target(canonical_answer)
    meta = {
        "equation_count": len(equations),
        "fallback_answer_only": len(equations) == 0,
        "equations_preview": equations[:8],
        "target_chars": len(target),
        "target_tokens": len(tokenizer(target, add_special_tokens=False)["input_ids"]),
    }
    return target, meta

def build_balanced_subset(records: list[dict], max_records: int | None, seed: int) -> list[dict]:
    if max_records is None or max_records >= len(records):
        return list(records)
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for rec in records:
        buckets[rec.get("type") or "unknown"].append(rec)
    for vals in buckets.values():
        rng.shuffle(vals)
    active_types = sorted(buckets)
    out = []
    cursor = 0
    while len(out) < max_records and active_types:
        t = active_types[cursor % len(active_types)]
        if buckets[t]:
            out.append(buckets[t].pop())
        if not buckets[t]:
            active_types.remove(t)
            cursor = 0
        else:
            cursor += 1
    rng.shuffle(out)
    return out

def split_source_disjoint(records: list[dict]) -> tuple[list[dict], list[dict], dict]:
    groups = defaultdict(list)
    for rec in records:
        groups[source_group_key(rec)].append(rec)
    buckets = defaultdict(list)
    for key, vals in groups.items():
        type_counts = Counter(v.get("type") or "unknown" for v in vals)
        dominant_type = sorted(type_counts.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
        buckets[dominant_type].append((key, vals))

    rng = random.Random(SEED)
    valid_keys = set()
    for type_name, items in buckets.items():
        rng.shuffle(items)
        n_valid = max(1, int(round(len(items) * SOURCE_VALID_GROUP_FRACTION))) if items else 0
        for key, _ in items[:n_valid]:
            valid_keys.add(key)

    source_train, source_valid = [], []
    for key, vals in groups.items():
        if key in valid_keys:
            source_valid.extend(vals)
        else:
            source_train.extend(vals)

    audit = {
        "source_group_fraction": SOURCE_VALID_GROUP_FRACTION,
        "total_records": len(records),
        "total_groups": len(groups),
        "source_train_records": len(source_train),
        "source_valid_records": len(source_valid),
        "source_train_groups": len(groups) - len(valid_keys),
        "source_valid_groups": len(valid_keys),
        "source_group_overlap": 0,
        "source_valid_by_type": dict(Counter(r.get("type") or "unknown" for r in source_valid).most_common()),
        "source_train_by_type": dict(Counter(r.get("type") or "unknown" for r in source_train).most_common()),
    }
    print("[source_split]", json.dumps(audit, ensure_ascii=False))
    return source_train, source_valid, audit

COMPACT_COVERAGE = {
    "version": "v9_compact_equation",
    "keep_all_equations": COMPACT_EQUATION_KEEP_ALL,
    "joiner": COMPACT_EQUATION_JOINER,
    "local_reason_max_tokens": LOCAL_REASON_MAX_TOKENS,
    "stages": {},
}

def _summarize_compact_meta(items: list[dict]) -> dict:
    counts = Counter(x.get("_compact_equation_count", 0) for x in items)
    target_tokens = [x.get("_target_tokens", 0) for x in items]
    n = len(items)
    return {
        "n": n,
        "fallback_answer_only": counts.get(0, 0),
        "one_equation": counts.get(1, 0),
        "multi_equation": sum(v for k, v in counts.items() if k >= 2),
        "equation_count_hist": dict(sorted((str(k), v) for k, v in counts.items())),
        "target_tokens_mean": sum(target_tokens) / n if n else 0.0,
        "target_tokens_max": max(target_tokens) if target_tokens else 0,
    }

def build_training_records_for_mode(records: list[dict], target_mode: str, stage_name: str, max_records: int | None = None) -> list[dict]:
    selected = build_balanced_subset(records, max_records, SEED + len(stage_name))
    out = []
    compact_items = []
    counts = Counter()
    for rec in selected:
        canonical = rec.get("_canonical_answer")
        if canonical is None:
            continue
        mode = target_mode
        if target_mode == "mixed_replay":
            mode = "compact_equation" if stable_fraction(f"{stage_name}:{rec['_source_id']}") < MIXED_COMPACT_RATIO else "answer_only"
        if mode == "answer_only":
            target = build_answer_only_target(canonical)
            meta = {
                "equation_count": 0,
                "fallback_answer_only": True,
                "target_tokens": len(tokenizer(target, add_special_tokens=False)["input_ids"]),
            }
        elif mode == "compact_equation":
            target, meta = build_compact_equation_target(rec.get("response_vi", ""), canonical)
            compact_items.append({**rec, "_compact_equation_count": meta["equation_count"], "_target_tokens": meta["target_tokens"]})
        else:
            raise ValueError(f"Unknown target_mode={target_mode}")
        counts[mode] += 1
        out.append({
            **rec,
            "response_vi": target,
            "_stage": stage_name,
            "_target_mode": mode,
            "_compact_equation_count": int(meta["equation_count"]),
            "_target_tokens": int(meta["target_tokens"]),
        })
    if compact_items:
        COMPACT_COVERAGE["stages"][stage_name] = _summarize_compact_meta(compact_items)
    print(f"[build:{stage_name}] mode={target_mode} selected={len(selected)} total={len(out)} counts={dict(counts)}")
    return out

valid_overlap_audit = save_valid_overlap_audit(train_records, valid_records, VALID_OVERLAP_AUDIT_PATH)
train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid_reference")
source_train_clean, source_valid_clean, source_split_audit = split_source_disjoint(train_clean)
source_valid_eval_records = build_balanced_subset(source_valid_clean, SOURCE_VALID_MAX_EVAL_RECORDS, SEED + 17)

train_stage_a = build_training_records_for_mode(source_train_clean, STAGE_A_TARGET_MODE, STAGE_A_NAME)
source_valid_stage_a = build_training_records_for_mode(source_valid_eval_records, STAGE_A_TARGET_MODE, STAGE_A_NAME + "_source_valid")

if RUN_STAGE_B:
    train_stage_b = build_training_records_for_mode(source_train_clean, STAGE_B_TARGET_MODE, STAGE_B_NAME, STAGE_B_MAX_RECORDS)
    source_valid_stage_b = build_training_records_for_mode(source_valid_eval_records, STAGE_B_TARGET_MODE, STAGE_B_NAME + "_source_valid")
else:
    train_stage_b, source_valid_stage_b = [], []

if RUN_STAGE_C:
    train_stage_c = build_training_records_for_mode(source_train_clean, STAGE_C_TARGET_MODE, STAGE_C_NAME, STAGE_C_MAX_RECORDS)
    source_valid_stage_c = build_training_records_for_mode(source_valid_eval_records, STAGE_C_TARGET_MODE, STAGE_C_NAME + "_source_valid")
else:
    train_stage_c, source_valid_stage_c = [], []

COMPACT_COVERAGE["source_split"] = source_split_audit
COMPACT_COVERAGE["source_valid_eval_n"] = len(source_valid_eval_records)
COMPACT_EQUATION_COVERAGE_PATH.write_text(json.dumps(COMPACT_COVERAGE, ensure_ascii=False, indent=2), encoding="utf-8")
print("[compact_coverage] wrote", COMPACT_EQUATION_COVERAGE_PATH)

print("\nExample targets:")
for name, records in [("stage_a", train_stage_a), ("stage_b", train_stage_b), ("stage_c", train_stage_c)]:
    if records:
        print(f"[{name}]", records[0]["response_vi"][:700])


In [ ]:
# ============================================================
# 4. Weighted SFT dataset: loss only on response tokens
# ============================================================
class SFTDataset(Dataset):
    """Pre-tokenize once and attach per-token loss weights.

    Prompt tokens keep label -100. Response tokens use equation/anchor/answer
    weights so the model can learn short reasoning without losing the final
    numeric-answer objective.
    """
    def __init__(self, records, tokenizer, max_length: int, desc: str = "train"):
        self.examples = []
        self.tok = tokenizer
        self.max_length = max_length
        for rec in tqdm(records, desc=f"tokenize:{desc}", leave=False):
            prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"])
            response = rec["response_vi"]
            canonical = rec.get("_canonical_answer")
            p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
            r_ids, r_weights = self._encode_response_with_weights(response, canonical)

            if len(p_ids) >= self.max_length - 1:
                p_ids = p_ids[-(self.max_length - 1):]
            budget = self.max_length - len(p_ids)
            if budget <= 0:
                r_ids = [SAFE_EOS_ID]
                r_weights = [EOS_TOKEN_WEIGHT]
            elif len(r_ids) > budget:
                # Keep the scoring-critical answer tail.
                r_ids = r_ids[-budget:]
                r_weights = r_weights[-budget:]

            ids = p_ids + r_ids
            labels = [-100] * len(p_ids) + r_ids
            weights = [0.0] * len(p_ids) + r_weights
            ids = [min(t, SAFE_EOS_ID) for t in ids]
            labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]
            self.examples.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids),
                "labels": labels,
                "loss_weights": weights,
                "length": len(ids),
            })

    def _encode_response_with_weights(self, response: str, canonical_answer: str | None):
        try:
            enc = self.tok(response, add_special_tokens=False, return_offsets_mapping=True)
            ids = enc["input_ids"]
            offsets = enc["offset_mapping"]
        except Exception:
            ids = self.tok(response, add_special_tokens=False)["input_ids"]
            return ids + [SAFE_EOS_ID], [EQUATION_TOKEN_WEIGHT] * len(ids) + [EOS_TOKEN_WEIGHT]

        weights = [EQUATION_TOKEN_WEIGHT] * len(ids)
        answer_match = None
        for m in FINAL_ANSWER_RE.finditer(response):
            answer_match = m
        if answer_match is not None:
            anchor_start, anchor_end = answer_match.start(1), answer_match.end(1)
            answer_start = answer_match.end(2)
            answer_end = len(response)
            if canonical_answer:
                tail = response[answer_start:]
                rel = tail.find(str(canonical_answer))
                if rel >= 0:
                    answer_start = answer_start + rel
                    answer_end = answer_start + len(str(canonical_answer))
            for i, (start, end) in enumerate(offsets):
                if end <= start:
                    continue
                if start < answer_end and end > answer_start:
                    weights[i] = FINAL_ANSWER_TOKEN_WEIGHT
                elif start < anchor_end and end > anchor_start:
                    weights[i] = FINAL_ANCHOR_TOKEN_WEIGHT
        return ids + [SAFE_EOS_ID], weights + [EOS_TOKEN_WEIGHT]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

@dataclass
class PadCollator:
    pad_id: int
    pad_to_multiple_of: int = 8

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        if self.pad_to_multiple_of:
            m = self.pad_to_multiple_of
            maxlen = ((maxlen + m - 1) // m) * m

        out = {"input_ids": [], "attention_mask": [], "labels": [], "loss_weights": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
            out["loss_weights"].append(x["loss_weights"] + [0.0] * pad)
        return {
            "input_ids": torch.tensor(out["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(out["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(out["labels"], dtype=torch.long),
            "loss_weights": torch.tensor(out["loss_weights"], dtype=torch.float32),
        }

def _probe_dataset(name: str, records: list[dict], max_length: int):
    if not records:
        print(f"{name}: skipped")
        return
    probe = SFTDataset(records[:1], tokenizer, max_length, desc=f"probe_{name}")[0]
    loss_tokens = sum(x != -100 for x in probe["labels"])
    weighted = sum(float(w) for w in probe["loss_weights"])
    print(f"{name} len/loss_tokens/weight_sum:", len(probe["input_ids"]), loss_tokens, round(weighted, 2))
    print(f"{name} tail:", decode_model_text(tokenizer, probe["input_ids"][-50:]))

_probe_dataset("stage_a", train_stage_a, MAX_LENGTH_STAGE_A)
_probe_dataset("stage_b", train_stage_b, MAX_LENGTH_STAGE_B)
_probe_dataset("stage_c", train_stage_c, MAX_LENGTH_STAGE_C)


In [ ]:
# ============================================================
# 5. Fast LoRA SFT with weighted final-answer loss
# ============================================================
def build_training_args(output_dir: Path, epochs: float, lr: float):
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        group_by_length=True,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "no"
    else:
        kwargs["evaluation_strategy"] = "no"
    if "optim" in sig.parameters and torch.cuda.is_available():
        kwargs["optim"] = "adamw_torch_fused"
    return TrainingArguments(**kwargs)

class WeightedCausalLMTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss_weights = inputs.pop("loss_weights", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view_as(shift_labels)
        valid_mask = shift_labels.ne(-100).float()
        if loss_weights is not None:
            shift_weights = loss_weights[..., 1:].to(token_loss.device).float()
            weighted_mask = valid_mask * shift_weights
        else:
            weighted_mask = valid_mask
        loss = (token_loss * weighted_mask).sum() / weighted_mask.sum().clamp_min(1.0)
        return (loss, outputs) if return_outputs else loss

def build_lora_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.use_cache = False

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        fan_in_fan_out=True,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

CHECKPOINT_SAVED = []

def _stage_slug(stage_name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(stage_name)).strip("_") or "stage"

def _checkpoint_label(stage_name: str, epoch_value: float | None, *, final: bool = False) -> str:
    prefix = _stage_slug(stage_name)
    if final:
        if epoch_value is None:
            return f"{prefix}_final"
        return f"{prefix}_final_epoch_{float(epoch_value):.2f}".replace(".", "p")
    if epoch_value is None:
        return f"{prefix}_epoch_unknown_{len(CHECKPOINT_SAVED)+1:02d}"
    ev = float(epoch_value)
    if abs(ev - round(ev)) < 1e-3:
        return f"{prefix}_epoch_{int(round(ev)):02d}"
    return f"{prefix}_epoch_{ev:.2f}".replace(".", "p")

def save_adapter_checkpoint(model, root_dir: Path, *, epoch_value: float | None, final: bool = False, stage_name: str = "stage"):
    root_dir.mkdir(parents=True, exist_ok=True)
    label = _checkpoint_label(stage_name, epoch_value, final=final)
    ckpt_dir = root_dir / label
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    meta = {
        "label": label,
        "stage_name": stage_name,
        "epoch": None if epoch_value is None else float(epoch_value),
        "final": bool(final),
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "loss_weights": {
            "equation": EQUATION_TOKEN_WEIGHT,
            "final_anchor": FINAL_ANCHOR_TOKEN_WEIGHT,
            "final_answer": FINAL_ANSWER_TOKEN_WEIGHT,
            "eos": EOS_TOKEN_WEIGHT,
        },
        "saved_at_unix": time.time(),
    }
    (ckpt_dir / "checkpoint_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    try:
        meta["sha256"] = sha256_dir(ckpt_dir)
        (ckpt_dir / "model_hash.txt").write_text(meta["sha256"] + "\n", encoding="utf-8")
    except Exception as exc:
        meta["sha256_error"] = repr(exc)
    CHECKPOINT_SAVED.append(meta | {"path": str(ckpt_dir)})
    print(f"[checkpoint] saved {label} -> {ckpt_dir}")
    return ckpt_dir

class SaveAdapterEveryEpochCallback(TrainerCallback):
    def __init__(self, root_dir: Path, stage_name: str):
        self.root_dir = Path(root_dir)
        self.stage_name = stage_name
        self._saved_labels = set()

    def on_epoch_end(self, args, state, control, **kwargs):
        model_obj = kwargs.get("model")
        if model_obj is None or state.epoch is None:
            return control
        label = _checkpoint_label(self.stage_name, float(state.epoch))
        if label in self._saved_labels:
            return control
        save_adapter_checkpoint(model_obj, self.root_dir, epoch_value=float(state.epoch), final=False, stage_name=self.stage_name)
        self._saved_labels.add(label)
        return control

def train_lora_stage(stage_name: str, model, train_records_for_stage, output_dir: Path, max_length: int, epochs: float, lr: float):
    if not train_records_for_stage or epochs <= 0:
        print(f"[train:{stage_name}] skipped")
        return model, 0.0
    print("\n" + "=" * 90)
    print(f"[train:{stage_name}] train={len(train_records_for_stage)} max_length={max_length} epochs={epochs} lr={lr}")
    train_ds = SFTDataset(train_records_for_stage, tokenizer, max_length, desc=stage_name)
    collator = PadCollator(SAFE_EOS_ID)

    eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
    print(f"[train:{stage_name}] eff_batch={eff_batch} steps/epoch={math.ceil(len(train_ds)/eff_batch)}")

    callbacks = []
    if SELECT_CHECKPOINTS_ON_SOURCE_VALID:
        callbacks.append(SaveAdapterEveryEpochCallback(CHECKPOINT_ROOT_DIR, stage_name))

    trainer = WeightedCausalLMTrainer(
        model=model,
        args=build_training_args(output_dir, epochs, lr),
        train_dataset=train_ds,
        data_collator=collator,
        callbacks=callbacks,
    )
    t0 = time.time()
    trainer.train()
    dt = time.time() - t0
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    model_hash = sha256_dir(output_dir)
    (output_dir / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
    print(f"[train:{stage_name}] wall={dt/60:.2f} min saved={output_dir} sha256={model_hash}")

    if SELECT_CHECKPOINTS_ON_SOURCE_VALID:
        epoch_float = float(epochs)
        integer_epoch_dir = CHECKPOINT_ROOT_DIR / _checkpoint_label(stage_name, epoch_float)
        if abs(epoch_float - round(epoch_float)) < 1e-3 and (integer_epoch_dir / "adapter_config.json").exists():
            print(f"[checkpoint] final epoch duplicates {integer_epoch_dir.name}; skip extra final checkpoint")
        else:
            save_adapter_checkpoint(model, CHECKPOINT_ROOT_DIR, epoch_value=epoch_float, final=True, stage_name=stage_name)
        saved_index_path = CHECKPOINT_ROOT_DIR / "checkpoint_index.json"
        saved_index_path.write_text(json.dumps(CHECKPOINT_SAVED, ensure_ascii=False, indent=2), encoding="utf-8")

    del trainer
    torch.cuda.empty_cache()
    return model, dt

model = build_lora_model()

stage_train_times = {}
model, stage_train_times[STAGE_A_NAME] = train_lora_stage(
    STAGE_A_NAME,
    model,
    train_stage_a,
    STAGE_A_OUTPUT_DIR,
    MAX_LENGTH_STAGE_A,
    STAGE_A_EPOCHS,
    STAGE_A_LR,
)

if RUN_STAGE_B:
    model, stage_train_times[STAGE_B_NAME] = train_lora_stage(
        STAGE_B_NAME,
        model,
        train_stage_b,
        SFT_OUTPUT_DIR,
        MAX_LENGTH_STAGE_B,
        STAGE_B_EPOCHS,
        STAGE_B_LR,
    )
else:
    SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(SFT_OUTPUT_DIR)
    tokenizer.save_pretrained(SFT_OUTPUT_DIR)
    (SFT_OUTPUT_DIR / "model_hash.txt").write_text(sha256_dir(SFT_OUTPUT_DIR) + "\n", encoding="utf-8")
    stage_train_times[STAGE_B_NAME] = 0.0

if RUN_STAGE_C:
    model, stage_train_times[STAGE_C_NAME] = train_lora_stage(
        STAGE_C_NAME,
        model,
        train_stage_c,
        STAGE_C_OUTPUT_DIR,
        MAX_LENGTH_STAGE_C,
        STAGE_C_EPOCHS,
        STAGE_C_LR,
    )
else:
    STAGE_C_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(STAGE_C_OUTPUT_DIR)
    tokenizer.save_pretrained(STAGE_C_OUTPUT_DIR)
    (STAGE_C_OUTPUT_DIR / "model_hash.txt").write_text(sha256_dir(STAGE_C_OUTPUT_DIR) + "\n", encoding="utf-8")
    stage_train_times[STAGE_C_NAME] = 0.0

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
(FINAL_OUTPUT_DIR / "model_hash.txt").write_text(sha256_dir(FINAL_OUTPUT_DIR) + "\n", encoding="utf-8")

print(f"[train:sft] total wall={sum(stage_train_times.values())/60:.2f} min")


In [ ]:
# ============================================================
# 6. Custom GRPO-lite reward tuning
# ============================================================
def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())

def has_anchor(text: str) -> bool:
    return extract_answer(text) is not None

def has_reason_equation(text: str) -> bool:
    before = ANSWER_TAIL_RE.split(text or "")[0]
    return bool(re.search(r"\d", before) and re.search(r"[+\-*/=×÷^]|\\frac|\\sqrt", before))

def reward_completion(completion: str, gold_num: float | None, gen_token_count: int) -> dict:
    completion = clean_decoded_artifacts(completion)
    pred_answer = extract_answer(completion)
    pred_num = parse_number(pred_answer)
    error_value = rel_error(pred_num, gold_num)
    answer_score = score_one(error_value, pred_answer is not None)
    reward_answer = answer_score / 10.0
    reward_anchor = 0.15 if pred_answer is not None else -0.20
    reward_len = 0.05 if gen_token_count <= 64 else (-0.05 if gen_token_count > 96 else 0.0)
    reward_reason = 0.05 if has_reason_equation(completion) else 0.0
    reward = max(0.0, min(1.2, reward_answer + reward_anchor + reward_len + reward_reason))
    return {
        "reward": reward,
        "reward_answer": reward_answer,
        "reward_anchor": reward_anchor,
        "reward_len": reward_len,
        "reward_reason": reward_reason,
        "answer_score": answer_score,
        "extractable": pred_answer is not None,
        "pred_answer": pred_answer,
        "pred_num": pred_num,
        "rel_error": error_value,
    }

def next_cyclic_batch(records: list[dict], cursor: int, batch_size: int):
    batch = []
    for j in range(batch_size):
        batch.append(records[(cursor + j) % len(records)])
    return batch, (cursor + batch_size) % len(records)

def generate_rl_group(model, batch_records: list[dict]):
    prompts = []
    for rec in batch_records:
        prompts.extend([build_prompt(rec)] * RL_GROUP_SIZE)
    model.eval()
    old_cache = getattr(model.config, "use_cache", False)
    model.config.use_cache = True
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=N_POSITIONS - RL_MAX_NEW_TOKENS,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        seqs = model.generate(
            **enc,
            max_new_tokens=RL_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=RL_TEMPERATURE,
            top_p=RL_TOP_P,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
        )
    model.config.use_cache = old_cache
    model.train()
    prompt_width = enc["input_ids"].shape[1]
    gen_ids = seqs[:, prompt_width:]
    texts = [decode_model_text(tokenizer, row) for row in gen_ids]
    gen_counts = (gen_ids != SAFE_EOS_ID).sum(dim=1).detach().cpu().tolist()
    return enc, seqs, gen_ids, texts, gen_counts, prompt_width

def completion_logprob_mean(model, seqs: torch.Tensor, input_attention: torch.Tensor, prompt_width: int):
    full_attention = torch.ones_like(seqs)
    full_attention[:, :prompt_width] = input_attention
    out = model(input_ids=seqs, attention_mask=full_attention)
    logits = out.logits[:, :-1, :]
    targets = seqs[:, 1:]
    token_logp = F.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    gen_ids = seqs[:, prompt_width:]
    gen_mask = (gen_ids != SAFE_EOS_ID).float()
    target_mask = torch.zeros_like(targets, dtype=torch.float32)
    target_mask[:, prompt_width - 1:] = gen_mask
    return (token_logp * target_mask).sum(dim=1) / target_mask.sum(dim=1).clamp(min=1.0)

def sft_replay_loss(model, records: list[dict]):
    replay = []
    for rec in records:
        canonical = rec.get("_canonical_answer")
        if canonical is not None:
            replay.append({**rec, "response_vi": build_answer_only_target(canonical)})
    ds = SFTDataset(replay, tokenizer, MAX_LENGTH_STAGE_A)
    collator = PadCollator(SAFE_EOS_ID)
    batch = collator([ds[i] for i in range(len(ds))])
    batch = {k: v.to(model.device) for k, v in batch.items()}
    return model(**batch).loss

def run_grpo_lite(model):
    if not RL_ENABLED or RL_MAX_STEPS <= 0:
        print("[rl] disabled; saving SFT model as final.")
        FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(FINAL_OUTPUT_DIR)
        tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
        model_hash = sha256_dir(FINAL_OUTPUT_DIR)
        (FINAL_OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
        summary = {
            "enabled": False,
            "reason": "RL_ENABLED is False or RL_MAX_STEPS <= 0",
            "final_dir": str(FINAL_OUTPUT_DIR),
            "sha256": model_hash,
        }
        RL_SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        return summary

    rl_records = build_balanced_subset(train_clean, RL_MAX_PROMPTS, SEED + 17)
    rng = random.Random(SEED + 19)
    rng.shuffle(rl_records)
    print(f"[rl] prompts={len(rl_records)} batch={RL_PROMPT_BATCH_SIZE} group={RL_GROUP_SIZE} steps={RL_MAX_STEPS}")

    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=RL_LR)
    cursor = 0
    log_rows = []
    t0 = time.time()
    RL_LOG_PATH.write_text("", encoding="utf-8")
    model.train()

    for step in tqdm(range(1, RL_MAX_STEPS + 1), desc="grpo-lite"):
        batch_records, cursor = next_cyclic_batch(rl_records, cursor, RL_PROMPT_BATCH_SIZE)
        enc, seqs, gen_ids, texts, gen_counts, prompt_width = generate_rl_group(model, batch_records)

        reward_infos = []
        for i, text in enumerate(texts):
            rec = batch_records[i // RL_GROUP_SIZE]
            reward_infos.append(reward_completion(text, rec.get("_gold_num"), gen_counts[i]))

        rewards = torch.tensor([x["reward"] for x in reward_infos], dtype=torch.float32, device=model.device)
        reward_group = rewards.view(len(batch_records), RL_GROUP_SIZE)
        advantages = (reward_group - reward_group.mean(dim=1, keepdim=True)).reshape(-1).detach()

        logprob_mean = completion_logprob_mean(model, seqs, enc["attention_mask"], prompt_width)
        rl_loss = -(advantages * logprob_mean).mean()
        replay_loss = sft_replay_loss(model, batch_records)
        total_loss = rl_loss + RL_SFT_REPLAY_COEF * replay_loss

        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), RL_GRAD_CLIP)
        optimizer.step()

        reward_mean = float(rewards.mean().detach().cpu())
        answer_reward_mean = sum(x["reward_answer"] for x in reward_infos) / len(reward_infos)
        extractable_rate = sum(x["extractable"] for x in reward_infos) / len(reward_infos)
        row = {
            "step": step,
            "loss": float(total_loss.detach().cpu()),
            "rl_loss": float(rl_loss.detach().cpu()),
            "sft_replay_loss": float(replay_loss.detach().cpu()),
            "reward_mean": reward_mean,
            "answer_reward_mean": answer_reward_mean,
            "extractable_rate": extractable_rate,
            "adv_abs_mean": float(advantages.abs().mean().detach().cpu()),
            "elapsed_min": (time.time() - t0) / 60,
        }
        log_rows.append(row)
        with RL_LOG_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        if step == 1 or step % 25 == 0:
            print("[rl]", json.dumps(row, ensure_ascii=False))

    FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(FINAL_OUTPUT_DIR)
    tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
    model_hash = sha256_dir(FINAL_OUTPUT_DIR)
    (FINAL_OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")

    tail = log_rows[-min(50, len(log_rows)):]
    summary = {
        "enabled": True,
        "steps": len(log_rows),
        "rl_max_prompts": RL_MAX_PROMPTS,
        "group_size": RL_GROUP_SIZE,
        "prompt_batch_size": RL_PROMPT_BATCH_SIZE,
        "reward_mean_last50": sum(x["reward_mean"] for x in tail) / len(tail) if tail else None,
        "answer_reward_mean_last50": sum(x["answer_reward_mean"] for x in tail) / len(tail) if tail else None,
        "extractable_rate_last50": sum(x["extractable_rate"] for x in tail) / len(tail) if tail else None,
        "wall_min": (time.time() - t0) / 60,
        "final_dir": str(FINAL_OUTPUT_DIR),
        "sha256": model_hash,
    }
    RL_SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[rl-summary]", json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

rl_summary = run_grpo_lite(model)

del model
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 7. Generation + source-valid checkpoint selection
# ============================================================
def has_peft_adapter(path_like) -> bool:
    p = Path(path_like)
    return p.exists() and (p / "adapter_config.json").exists()

def load_model_for_generation(adapter_dir: Path):
    dtype = torch.float16 if (INFER_FP16 and torch.cuda.is_available()) else torch.float32
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, local_files_only=True)
    base.config.pad_token_id = SAFE_EOS_ID
    base.config.eos_token_id = SAFE_EOS_ID
    if has_peft_adapter(adapter_dir):
        print(f"[infer] base + adapter: {adapter_dir}")
        gen_model = PeftModel.from_pretrained(base, str(adapter_dir), local_files_only=True)
        try:
            gen_model = gen_model.merge_and_unload()
            print("[infer] merged LoRA adapter")
        except Exception as exc:
            print("[infer] merge failed; using PEFT wrapper:", repr(exc))
    else:
        print(f"[infer] no adapter at {adapter_dir}; using base model")
        gen_model = base
    device = "cuda" if torch.cuda.is_available() else "cpu"
    gen_model.to(device)
    gen_model.eval()
    return gen_model

class StopOnAnswerLine(StoppingCriteria):
    """Stop after the final answer line has had enough continuation room."""
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID, patience_tokens: int = 16):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_answer = re.compile(r"đáp\s*án\s*l[àa]\s*[:：]\s*-?\d", re.IGNORECASE)
        self._matched_at = None

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 4:
            return False
        text = decode_model_text(self.tok, gen_tail)
        m = self.re_answer.search(text)
        if not m:
            return False
        if self._matched_at is None:
            self._matched_at = gen_tail.numel()
        if "\n" in text[m.end():]:
            return True
        if gen_tail.numel() - self._matched_at >= self.patience:
            return True
        return False

@torch.inference_mode()
def generate_outputs(adapter_dir: Path, records: list[dict], output_path: Path, *, max_new_tokens: int = MAX_NEW_TOKENS, num_beams: int = NUM_BEAMS):
    gen_tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
    gen_tok.pad_token_id = SAFE_EOS_ID
    gen_tok.eos_token_id = SAFE_EOS_ID
    gen_tok.padding_side = "left"
    gen_tok.truncation_side = "left"
    gen_model = load_model_for_generation(adapter_dir)
    device = next(gen_model.parameters()).device
    outputs = []
    n_pos = int(getattr(gen_model.config, "n_positions", getattr(gen_model.config, "max_position_embeddings", 1024)))
    vocab_n = gen_model.get_input_embeddings().num_embeddings
    for idx, rec in enumerate(tqdm(records, desc=f"generate:{Path(adapter_dir).name}")):
        prompt = build_prompt(rec)
        prompt_budget = max(8, n_pos - max_new_tokens)
        full_ids = gen_tok(prompt, add_special_tokens=False)["input_ids"]
        if len(full_ids) > prompt_budget:
            full_ids = full_ids[-prompt_budget:]
        full_ids = [min(t, vocab_n - 1) for t in full_ids]
        ids = torch.tensor([full_ids], dtype=torch.long, device=device)
        attn = torch.ones_like(ids)
        prompt_len = ids.shape[1]
        eff_new = max(8, min(max_new_tokens, n_pos - prompt_len))
        gen_kwargs = dict(
            input_ids=ids,
            attention_mask=attn,
            max_new_tokens=eff_new,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            stopping_criteria=StoppingCriteriaList([StopOnAnswerLine(gen_tok, prompt_len=prompt_len)]),
        )
        if num_beams and num_beams > 1:
            gen_kwargs.update(dict(num_beams=num_beams, do_sample=False, early_stopping=True, length_penalty=LENGTH_PENALTY))
        else:
            gen_kwargs.update(dict(num_beams=1, do_sample=False))
        seqs = gen_model.generate(**gen_kwargs)
        text = decode_model_text(gen_tok, seqs[0, prompt_len:])
        if SANITIZE_TO_ANSWER_ONLY:
            text = sanitize_model_output(text)
        outputs.append({
            "id": rec.get("id", idx),
            "query_vi": rec.get("query_vi", ""),
            "type": rec.get("type"),
            "model_output": text.strip(),
        })
    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[infer] wrote {len(outputs)} rows -> {output_path}")
    del gen_model
    torch.cuda.empty_cache()
    return outputs

def _adapter_sort_key(path: Path):
    meta_path = path / "checkpoint_meta.json"
    epoch = 10**9
    final = False
    stage = path.name
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            epoch = float(meta.get("epoch") or 10**9)
            final = bool(meta.get("final"))
            stage = str(meta.get("stage_name") or path.name)
        except Exception:
            pass
    return (stage, epoch, int(final), path.name)

def list_stage_checkpoints() -> list[Path]:
    candidates = []
    if CHECKPOINT_ROOT_DIR.exists():
        candidates.extend([p for p in CHECKPOINT_ROOT_DIR.iterdir() if p.is_dir() and has_peft_adapter(p)])
    if not candidates:
        for p in [STAGE_A_OUTPUT_DIR, SFT_OUTPUT_DIR, STAGE_C_OUTPUT_DIR, FINAL_OUTPUT_DIR]:
            if has_peft_adapter(p):
                candidates.append(p)
    dedup = []
    seen = set()
    for p in candidates:
        key = str(p.resolve())
        if key not in seen:
            dedup.append(p)
            seen.add(key)
    return sorted(dedup, key=_adapter_sort_key)

def _score_tuple(summary: dict, order: int):
    buckets = summary.get("buckets", {})
    exact10 = buckets.get("10", buckets.get(10, 0))
    raw = summary.get("raw_score", -1)
    extractable = summary.get("extractable", -1)
    order_term = -order if CHECKPOINT_TIE_BREAK == "earlier_epoch" else order
    return (raw, exact10, extractable, order_term)

def select_best_checkpoint_on_source_valid(records_for_selection: list[dict]):
    global CHECKPOINT_SELECTION
    ckpts = list_stage_checkpoints()
    if not ckpts:
        print("[select] no adapter checkpoints found; keeping current FINAL_OUTPUT_DIR")
        CHECKPOINT_SELECTION = {
            "enabled": True,
            "selection_split": "source_valid_from_train",
            "status": "no_checkpoints_found",
            "final_output_dir": str(FINAL_OUTPUT_DIR),
        }
        SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(CHECKPOINT_SELECTION, ensure_ascii=False, indent=2), encoding="utf-8")
        return CHECKPOINT_SELECTION

    CHECKPOINT_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    eval_records = records_for_selection
    print(f"[select] evaluating {len(ckpts)} checkpoints on {len(eval_records)} source-valid rows")

    entries = []
    best_entry = None
    best_key = None
    for order, ckpt in enumerate(ckpts):
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", ckpt.name)
        out_path = CHECKPOINT_EVAL_DIR / f"source_valid_output_{order:02d}_{safe_name}.json"
        report_path = CHECKPOINT_EVAL_DIR / f"source_valid_report_{order:02d}_{safe_name}.json"
        _ = generate_outputs(
            ckpt,
            eval_records,
            out_path,
            max_new_tokens=CHECKPOINT_EVAL_MAX_NEW_TOKENS,
            num_beams=CHECKPOINT_EVAL_NUM_BEAMS,
        )
        rep = save_eval_report(out_path, eval_records, report_path)
        summary = rep["summary"]
        meta = {}
        meta_path = ckpt / "checkpoint_meta.json"
        if meta_path.exists():
            try:
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
            except Exception as exc:
                meta = {"meta_error": repr(exc)}
        entry = {
            "order": order,
            "label": ckpt.name,
            "adapter_dir": str(ckpt),
            "output_path": str(out_path),
            "report_path": str(report_path),
            "summary": summary,
            "meta": meta,
        }
        key = _score_tuple(summary, order)
        entry["selection_key"] = list(key)
        entries.append(entry)
        print(f"[select] {ckpt.name}: raw={summary['raw_score']} exact10={summary['buckets'].get('10')} extractable={summary['extractable']} key={key}")
        if best_key is None or key > best_key:
            best_key = key
            best_entry = entry
        if not KEEP_CHECKPOINT_EVAL_OUTPUTS:
            try:
                out_path.unlink()
                entry["output_path_deleted"] = True
            except Exception:
                pass

    if best_entry is None:
        raise RuntimeError("Checkpoint selection failed: no best checkpoint")

    selected_dir = Path(best_entry["adapter_dir"])
    if FINAL_OUTPUT_DIR.exists():
        shutil.rmtree(FINAL_OUTPUT_DIR)
    shutil.copytree(selected_dir, FINAL_OUTPUT_DIR)
    final_hash = sha256_dir(FINAL_OUTPUT_DIR)
    (FINAL_OUTPUT_DIR / "model_hash.txt").write_text(final_hash + "\n", encoding="utf-8")

    CHECKPOINT_SELECTION = {
        "enabled": True,
        "selection_split": "source_valid_from_train",
        "selection_metric": "max(raw_score, exact10, extractable, tie_break)",
        "tie_break": CHECKPOINT_TIE_BREAK,
        "status": "selected",
        "eval_n": len(eval_records),
        "num_checkpoints": len(entries),
        "selected": best_entry,
        "selected_adapter_dir": str(selected_dir),
        "final_output_dir": str(FINAL_OUTPUT_DIR),
        "final_sha256": final_hash,
        "all_checkpoints": entries,
    }
    CHECKPOINT_SELECTION_REPORT_PATH.write_text(json.dumps(CHECKPOINT_SELECTION, ensure_ascii=False, indent=2), encoding="utf-8")
    SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(CHECKPOINT_SELECTION["selected"], ensure_ascii=False, indent=2), encoding="utf-8")
    print("[select] selected:", best_entry["label"], best_entry["summary"])
    print("[select] copied to:", FINAL_OUTPUT_DIR, "sha256=", final_hash)
    return CHECKPOINT_SELECTION

CHECKPOINT_SELECTION = {"enabled": False, "status": "not_run"}

if RUN_MODE == "phase1":
    if SELECT_CHECKPOINTS_ON_SOURCE_VALID:
        CHECKPOINT_SELECTION = select_best_checkpoint_on_source_valid(source_valid_eval_records)

    if (
        CHECKPOINT_SELECTION.get("status") == "selected"
        and CHECKPOINT_SELECTION.get("eval_n") == len(source_valid_eval_records)
        and CHECKPOINT_EVAL_MAX_NEW_TOKENS == MAX_NEW_TOKENS
        and CHECKPOINT_EVAL_NUM_BEAMS == NUM_BEAMS
    ):
        selected = CHECKPOINT_SELECTION["selected"]
        shutil.copyfile(selected["output_path"], SOURCE_VALID_OUTPUT_PATH)
        shutil.copyfile(selected["report_path"], SOURCE_VALID_REPORT_PATH)
        source_rep = json.loads(SOURCE_VALID_REPORT_PATH.read_text(encoding="utf-8"))
        print("[source_valid:selected] reused checkpoint-eval output/report")
    else:
        _ = generate_outputs(FINAL_OUTPUT_DIR, source_valid_eval_records, SOURCE_VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
        source_rep = save_eval_report(SOURCE_VALID_OUTPUT_PATH, source_valid_eval_records, SOURCE_VALID_REPORT_PATH)
    print("[source_valid:selected]", source_rep["summary"])

    # valid.json is report-only for backward comparison with earlier notebooks.
    if VALID_FILE.exists():
        _ = generate_outputs(FINAL_OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
        valid_rep = save_eval_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
        print("[valid_json:reference_only]", valid_rep["summary"])
        print("Reference valid.json Score /10:", valid_rep["summary"]["score_10"])
    else:
        print("[valid_json] skipped; file not found")

elif RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"RUN_MODE='phase2' but missing {TEST_FILE}")
    if SELECT_CHECKPOINTS_ON_SOURCE_VALID:
        CHECKPOINT_SELECTION = select_best_checkpoint_on_source_valid(source_valid_eval_records)
    else:
        print("[phase2] checkpoint selection skipped; using FINAL_OUTPUT_DIR from training/final adapter")
    test_records = load_records(TEST_FILE)
    _ = generate_outputs(FINAL_OUTPUT_DIR, test_records, TEST_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    print("[phase2] wrote", TEST_OUTPUT_PATH)
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE}")


In [ ]:
# ============================================================
# 8. Output manifest
# ============================================================
def _path_exists_str(p):
    try:
        return Path(p).exists()
    except Exception:
        return False

def _maybe_json_summary(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("summary")
    except Exception as exc:
        return {"error": repr(exc)}

manifest = {
    "notebook_version": "v9_compact_equation",
    "run_mode": RUN_MODE,
    "created_at_unix": time.time(),
    "use_kd": USE_KD,
    "data": {
        "train_file": str(TRAIN_FILE),
        "valid_file": str(VALID_FILE),
        "test_file": str(TEST_FILE),
        "valid_overlap_audit": str(VALID_OVERLAP_AUDIT_PATH),
        "source_split_audit": globals().get("source_split_audit"),
        "source_valid_eval_n": len(globals().get("source_valid_eval_records", [])),
    },
    "config": {
        "prompt_template": PROMPT_TEMPLATE,
        "safe_eos_id": SAFE_EOS_ID,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_valid_samples": MAX_VALID_SAMPLES,
        "drop_exact_duplicates": DROP_EXACT_DUPLICATES,
        "drop_non_extractable": DROP_NON_EXTRACTABLE,
        "source_valid_group_fraction": SOURCE_VALID_GROUP_FRACTION,
        "source_valid_max_eval_records": SOURCE_VALID_MAX_EVAL_RECORDS,
        "source_group_key_fields": SOURCE_GROUP_KEY_FIELDS,
        "stage_a_name": STAGE_A_NAME,
        "stage_a_target_mode": STAGE_A_TARGET_MODE,
        "stage_a_epochs": STAGE_A_EPOCHS,
        "stage_a_lr": STAGE_A_LR,
        "max_length_stage_a": MAX_LENGTH_STAGE_A,
        "run_stage_b": RUN_STAGE_B,
        "stage_b_target_mode": STAGE_B_TARGET_MODE,
        "stage_b_epochs": STAGE_B_EPOCHS,
        "stage_b_lr": STAGE_B_LR,
        "run_stage_c": RUN_STAGE_C,
        "stage_c_target_mode": STAGE_C_TARGET_MODE,
        "stage_c_epochs": STAGE_C_EPOCHS,
        "stage_c_lr": STAGE_C_LR,
        "mixed_compact_ratio": MIXED_COMPACT_RATIO,
        "compact_equation_keep_all": COMPACT_EQUATION_KEEP_ALL,
        "local_reason_max_tokens": LOCAL_REASON_MAX_TOKENS,
        "loss_weights": {
            "equation": EQUATION_TOKEN_WEIGHT,
            "final_anchor": FINAL_ANCHOR_TOKEN_WEIGHT,
            "final_answer": FINAL_ANSWER_TOKEN_WEIGHT,
            "eos": EOS_TOKEN_WEIGHT,
        },
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "decode_batch_size": DECODE_BATCH_SIZE,
        "no_repeat_ngram": NO_REPEAT_NGRAM,
        "repetition_penalty": REPETITION_PENALTY,
        "length_penalty": LENGTH_PENALTY,
        "sanitize_to_answer_only": SANITIZE_TO_ANSWER_ONLY,
        "infer_fp16": INFER_FP16,
        "select_checkpoints_on_source_valid": SELECT_CHECKPOINTS_ON_SOURCE_VALID,
        "checkpoint_eval_num_beams": CHECKPOINT_EVAL_NUM_BEAMS,
        "checkpoint_eval_max_new_tokens": CHECKPOINT_EVAL_MAX_NEW_TOKENS,
        "checkpoint_tie_break": CHECKPOINT_TIE_BREAK,
    },
    "dirs": {
        "stage_a_output_dir": str(STAGE_A_OUTPUT_DIR),
        "sft_output_dir": str(SFT_OUTPUT_DIR),
        "stage_c_output_dir": str(STAGE_C_OUTPUT_DIR),
        "final_output_dir": str(FINAL_OUTPUT_DIR),
        "checkpoint_root_dir": str(CHECKPOINT_ROOT_DIR),
        "checkpoint_eval_dir": str(CHECKPOINT_EVAL_DIR),
    },
    "outputs": {
        "source_valid_output": str(SOURCE_VALID_OUTPUT_PATH),
        "source_valid_report": str(SOURCE_VALID_REPORT_PATH),
        "valid_output": str(VALID_OUTPUT_PATH),
        "valid_report": str(VALID_REPORT_PATH),
        "compact_equation_coverage": str(COMPACT_EQUATION_COVERAGE_PATH),
        "checkpoint_selection_report": str(CHECKPOINT_SELECTION_REPORT_PATH),
        "selected_checkpoint_info": str(SELECTED_CHECKPOINT_INFO_PATH),
        "test_predictions": str(TEST_OUTPUT_PATH),
    },
    "checkpoint_selection": globals().get("CHECKPOINT_SELECTION", {"enabled": False, "status": "missing_global"}),
    "source_valid_summary": _maybe_json_summary(SOURCE_VALID_REPORT_PATH),
    "reference_valid_summary": _maybe_json_summary(VALID_REPORT_PATH),
    "compact_equation_coverage": globals().get("COMPACT_COVERAGE"),
    "stage_train_times": globals().get("stage_train_times"),
    "final_output_dir_exists": _path_exists_str(FINAL_OUTPUT_DIR),
}
manifest_path = WORKING_DIR / "v9_compact_equation_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("[manifest] wrote", manifest_path)
